# 00 — Explore & Profile (Bronze)

**Tickets:** I-02  
**Purpose:** Initial EDA of the raw NYC Yellow Taxi data — row counts, null rates, distributions, outliers.

---

## Setup

In [0]:
from pyspark.sql import functions as F

# On Databricks `spark` is injected automatically.
# Uncomment the two lines below only when running locally outside Databricks.
# spark = SparkSession.builder.appName("nyc_taxi_bronze_eda").getOrCreate()

print(f"Spark version: {spark.version}")

## Row counts & schema

In [0]:
from src.constants import BRONZE_TABLE

df = spark.read.table(BRONZE_TABLE)

print(f"Rows    : {df.count():,}")
print(f"Columns : {len(df.columns)}")

df.printSchema()

# Visual spot-check of raw values.
display(df.limit(10))

## Null & duplicate analysis

In [0]:
total_rows = df.count()

# --- Null counts ---
null_row = df.select(
    [F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in df.columns]
).collect()[0]

null_summary = (
    spark.createDataFrame(
        [(c, int(null_row[c])) for c in df.columns],
        ["column_name", "null_count"],
    )
    .withColumn("null_pct", F.round(F.col("null_count") / total_rows * 100, 2))
    .orderBy(F.col("null_count").desc())
)

print("=== Null counts per column ===")
display(null_summary)

# --- Duplicate rows ---
distinct_rows = df.distinct().count()
duplicate_count = total_rows - distinct_rows
print(f"\nTotal rows    : {total_rows:,}")
print(f"Distinct rows : {distinct_rows:,}")
print(
    f"Duplicates    : {duplicate_count:,}  ({duplicate_count / total_rows * 100:.2f}%)"
)

## Value distributions & outliers

In [0]:
# --- Numeric summary statistics ---
# Bronze columns may be ingested as strings; cast the known numeric fields for describe().
NUMERIC_COLS = [
    "passenger_count",
    "trip_distance",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "fare_amount",
    "extra",
    "mta_tax",
    "improvement_surcharge",
    "tip_amount",
    "tolls_amount",
    "total_amount",
]

numeric_df = df.select(
    [F.col(c).cast("double").alias(c) for c in NUMERIC_COLS if c in df.columns]
)

print("=== Numeric column statistics ===")
display(numeric_df.describe())

### Categorical value distributions

Inspect every low-cardinality field to catch unexpected codes or encoding issues in the raw data.

In [0]:
CATEGORICAL_COLS = {
    "VendorID": {1: "Creative Mobile Technologies", 2: "VeriFone Inc."},
    "RateCodeID": {
        1: "Standard",
        2: "JFK",
        3: "Newark",
        4: "Nassau/Westchester",
        5: "Negotiated",
        6: "Group ride",
    },
    "store_and_fwd_flag": {},  # Y / N
    "payment_type": {
        1: "Credit card",
        2: "Cash",
        3: "No charge",
        4: "Dispute",
        5: "Unknown",
        6: "Voided trip",
    },
}

for col_name in CATEGORICAL_COLS:
    if col_name not in df.columns:
        print(f"Column not found: {col_name}")
        continue
    print(f"\n=== {col_name} ===")
    display(
        df.groupBy(col_name)
        .count()
        .withColumn("pct", F.round(F.col("count") / total_rows * 100, 2))
        .orderBy(F.col("count").desc())
    )

### Outlier detection

Flag business-rule violations that should be treated as corrupt data in the Silver cleaning step (I-03/I-04).

In [0]:
def cast(col_name: str) -> F.Column:
    """Cast a Bronze column to double for numeric comparisons."""
    return F.col(col_name).cast("double")


outlier_summary = df.agg(
    F.count("*").alias("total_rows"),
    # Distance issues
    F.sum(F.when(cast("trip_distance") == 0, 1).otherwise(0)).alias("zero_distance"),
    F.sum(F.when(cast("trip_distance") < 0, 1).otherwise(0)).alias("negative_distance"),
    F.sum(F.when(cast("trip_distance") > 100, 1).otherwise(0)).alias(
        "distance_over_100mi"
    ),
    # Fare issues
    F.sum(F.when(cast("fare_amount") <= 0, 1).otherwise(0)).alias("zero_or_neg_fare"),
    F.sum(F.when(cast("fare_amount") > 500, 1).otherwise(0)).alias("fare_over_500"),
    # Total amount issues
    F.sum(F.when(cast("total_amount") < 0, 1).otherwise(0)).alias(
        "negative_total_amount"
    ),
    # Passenger issues
    F.sum(F.when(cast("passenger_count") == 0, 1).otherwise(0)).alias(
        "zero_passengers"
    ),
    F.sum(F.when(cast("passenger_count") > 6, 1).otherwise(0)).alias(
        "passengers_over_6"
    ),
    # Tip on non-card payments (tip should only be auto-populated for credit card)
    F.sum(
        F.when((cast("payment_type") != 1) & (cast("tip_amount") > 0), 1).otherwise(0)
    ).alias("tip_on_non_card_payment"),
)

display(outlier_summary)

### Temporal distributions

Understand when trips occur — drives demand heatmaps (BQ-1) and is a key feature for ML fare prediction (BQ-3).  
> **Note:** Bronze datetimes may be raw strings; `to_timestamp` handles standard YYYY-MM-DD HH:mm:ss format.

In [0]:
pickup_ts = F.to_timestamp("tpep_pickup_datetime")

temporal_df = df.withColumn("pickup_ts", pickup_ts)

# Date range
print("=== Pickup date range ===")
display(
    temporal_df.agg(
        F.min("pickup_ts").alias("earliest_pickup"),
        F.max("pickup_ts").alias("latest_pickup"),
    )
)

# Trips by hour of day
print("\n=== Trips by hour of day ===")
display(
    temporal_df.groupBy(F.hour("pickup_ts").alias("hour_of_day"))
    .count()
    .orderBy("hour_of_day")
)

# Trips by day of week (1 = Sunday … 7 = Saturday in Spark)
print("\n=== Trips by day of week ===")
display(
    temporal_df.groupBy(F.dayofweek("pickup_ts").alias("day_of_week"))
    .count()
    .orderBy("day_of_week")
)

# Trips by calendar month (useful for spotting seasonal patterns or bad data)
print("\n=== Trips by month ===")
display(
    temporal_df.groupBy(
        F.year("pickup_ts").alias("year"),
        F.month("pickup_ts").alias("month"),
    )
    .count()
    .orderBy("year", "month")
)

### GPS coordinate quality

NYC bounding box (approximate): longitude **[-74.3, -73.7]**, latitude **[40.4, 40.95]**.  
Coordinates at (0, 0) or far outside this box are corrupt and unusable for demand heatmaps (BQ-1).

In [0]:
# --- GPS coordinate quality ---
NYC_LON_MIN, NYC_LON_MAX = -74.3, -73.7
NYC_LAT_MIN, NYC_LAT_MAX = 40.4, 40.95

gps_quality = df.agg(
    F.count("*").alias("total_rows"),
    # Zero / null coordinates
    F.sum(F.when(
        (F.col("pickup_longitude") == 0) & (F.col("pickup_latitude") == 0), 1
    ).otherwise(0)).alias("pickup_zero_coords"),
    F.sum(F.when(
        (F.col("dropoff_longitude") == 0) & (F.col("dropoff_latitude") == 0), 1
    ).otherwise(0)).alias("dropoff_zero_coords"),
    # Outside NYC bounding box (excluding zeros already counted)
    F.sum(F.when(
        ~(
            (F.col("pickup_longitude").between(NYC_LON_MIN, NYC_LON_MAX)) &
            (F.col("pickup_latitude").between(NYC_LAT_MIN, NYC_LAT_MAX))
        ) &
        ~((F.col("pickup_longitude") == 0) & (F.col("pickup_latitude") == 0)),
        1
    ).otherwise(0)).alias("pickup_outside_nyc"),
    F.sum(F.when(
        ~(
            (F.col("dropoff_longitude").between(NYC_LON_MIN, NYC_LON_MAX)) &
            (F.col("dropoff_latitude").between(NYC_LAT_MIN, NYC_LAT_MAX))
        ) &
        ~((F.col("dropoff_longitude") == 0) & (F.col("dropoff_latitude") == 0)),
        1
    ).otherwise(0)).alias("dropoff_outside_nyc"),
)

print("=== GPS coordinate quality ===")
display(gps_quality)

# Show the actual coordinate ranges for context
print("\n=== Coordinate ranges (non-zero only) ===")
coord_ranges = df.filter(
    ~((F.col("pickup_longitude") == 0) & (F.col("pickup_latitude") == 0))
).agg(
    F.min("pickup_longitude").alias("min_pickup_lon"),
    F.max("pickup_longitude").alias("max_pickup_lon"),
    F.min("pickup_latitude").alias("min_pickup_lat"),
    F.max("pickup_latitude").alias("max_pickup_lat"),
    F.min("dropoff_longitude").alias("min_dropoff_lon"),
    F.max("dropoff_longitude").alias("max_dropoff_lon"),
    F.min("dropoff_latitude").alias("min_dropoff_lat"),
    F.max("dropoff_latitude").alias("max_dropoff_lat"),
)
display(coord_ranges)

### Trip duration analysis

Compute `dropoff − pickup` to detect impossible trips: negative durations, zero-second trips, and multi-day outliers.

In [0]:
# --- Trip duration analysis ---
duration_df = df.withColumn(
    "trip_duration_min",
    (F.col("tpep_dropoff_datetime").cast("long") - F.col("tpep_pickup_datetime").cast("long")) / 60.0
)

duration_quality = duration_df.agg(
    F.count("*").alias("total_rows"),
    F.min("trip_duration_min").alias("min_duration_min"),
    F.max("trip_duration_min").alias("max_duration_min"),
    F.mean("trip_duration_min").alias("avg_duration_min"),
    F.sum(F.when(F.col("trip_duration_min") < 0, 1).otherwise(0)).alias("negative_duration"),
    F.sum(F.when(F.col("trip_duration_min") == 0, 1).otherwise(0)).alias("zero_duration"),
    F.sum(F.when(F.col("trip_duration_min") < 1, 1).otherwise(0)).alias("under_1_min"),
    F.sum(F.when(F.col("trip_duration_min") > 180, 1).otherwise(0)).alias("over_3_hours"),
    F.sum(F.when(F.col("trip_duration_min") > 1440, 1).otherwise(0)).alias("over_24_hours"),
)

print("=== Trip duration quality ===")
display(duration_quality)

# Duration distribution buckets
print("\n=== Duration distribution (minutes) ===")
display(
    duration_df.withColumn(
        "duration_bucket",
        F.when(F.col("trip_duration_min") < 0, "< 0 (negative)")
        .when(F.col("trip_duration_min") == 0, "0 (zero)")
        .when(F.col("trip_duration_min") <= 5, "0–5")
        .when(F.col("trip_duration_min") <= 15, "5–15")
        .when(F.col("trip_duration_min") <= 30, "15–30")
        .when(F.col("trip_duration_min") <= 60, "30–60")
        .when(F.col("trip_duration_min") <= 180, "60–180")
        .when(F.col("trip_duration_min") <= 1440, "180–1440")
        .otherwise("> 1440 (> 24h)")
    )
    .groupBy("duration_bucket")
    .agg(F.count("*").alias("count"))
    .withColumn("pct", F.round(F.col("count") / total_rows * 100, 3))
    .orderBy("duration_bucket")
)

### Extreme monetary outliers

The `describe()` revealed extreme values well beyond current thresholds:  
`tip_amount` max ≈ $3.95M, `total_amount` max ≈ $3.95M, `extra` ranges from −$79 to $999.99.

In [0]:
# --- Extreme monetary outlier analysis ---
# Tip extremes
print("=== Tip amount extremes ===")
tip_extremes = df.agg(
    F.sum(F.when(F.col("tip_amount") < 0, 1).otherwise(0)).alias("negative_tip"),
    F.sum(F.when(F.col("tip_amount") > 100, 1).otherwise(0)).alias("tip_over_100"),
    F.sum(F.when(F.col("tip_amount") > 1000, 1).otherwise(0)).alias("tip_over_1000"),
    F.max("tip_amount").alias("max_tip"),
    F.min("tip_amount").alias("min_tip"),
)
display(tip_extremes)

# Total amount extremes
print("\n=== Total amount extremes ===")
total_extremes = df.agg(
    F.sum(F.when(F.col("total_amount") > 500, 1).otherwise(0)).alias("total_over_500"),
    F.sum(F.when(F.col("total_amount") > 1000, 1).otherwise(0)).alias("total_over_1000"),
    F.max("total_amount").alias("max_total"),
)
display(total_extremes)

# Extra surcharge (should be 0, 0.5, or 1.0 only)
print("\n=== Extra surcharge distribution ===")
display(
    df.groupBy("extra")
    .count()
    .withColumn("pct", F.round(F.col("count") / total_rows * 100, 3))
    .orderBy(F.col("count").desc())
    .limit(15)
)

### Tip distribution by payment type

Cash tips aren't recorded electronically, so `tip_amount` should be near-zero for non-credit-card payments.  
This directly informs BQ-4 (tipping behaviour) and ML tip-prediction feature engineering.

In [0]:
# --- Tip distribution by payment type ---
PAYMENT_LABELS = {
    1: "Credit card",
    2: "Cash",
    3: "No charge",
    4: "Dispute",
    5: "Unknown",
    6: "Voided trip",
}

tip_by_payment = (
    df.groupBy("payment_type")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.mean("tip_amount"), 2).alias("avg_tip"),
        F.round(F.expr("percentile(tip_amount, 0.5)"), 2).alias("median_tip"),
        F.max("tip_amount").alias("max_tip"),
        F.sum(F.when(F.col("tip_amount") > 0, 1).otherwise(0)).alias("trips_with_tip"),
        F.sum(F.when(F.col("tip_amount") == 0, 1).otherwise(0)).alias("trips_zero_tip"),
    )
    .withColumn("tip_rate_pct", F.round(F.col("trips_with_tip") / F.col("trip_count") * 100, 2))
    .orderBy(F.col("trip_count").desc())
)

print("=== Tip behaviour by payment type ===")
display(tip_by_payment)

# Tip amount distribution for credit-card payments only (BQ-4 / ML feature)
print("\n=== Credit-card tip distribution (buckets) ===")
display(
    df.filter(F.col("payment_type") == 1)
    .withColumn(
        "tip_bucket",
        F.when(F.col("tip_amount") == 0, "$0")
        .when(F.col("tip_amount") <= 2, "$0–2")
        .when(F.col("tip_amount") <= 5, "$2–5")
        .when(F.col("tip_amount") <= 10, "$5–10")
        .when(F.col("tip_amount") <= 20, "$10–20")
        .otherwise("> $20")
    )
    .groupBy("tip_bucket")
    .agg(F.count("*").alias("count"))
    .withColumn("pct", F.round(F.col("count") / F.lit(61741228) * 100, 2))
    .orderBy("tip_bucket")
)

## Findings & Assumptions

Observations from the cells above (94,497,690 total rows). This feeds directly into I-07 (data quality log).

| # | Finding | Affected column(s) | Rows | % | Recommended action (I-03/I-04) |
|---|---------|-------------------|------|---|--------------------------------|
| 1 | **Schema evolution — column casing**: The 2016 CSV files use `RatecodeID` (lowercase "c") while the 2015 file uses `RateCodeID` (uppercase "C"). This mismatch caused `RateCodeID` to be NULL for all 69M 2016 rows, with the value rescued into `_rescued_data` as JSON (`{"RatecodeID":"...", "_file_path":"..."}`). **100% of rescued rows contain a valid `RatecodeID`** — full recovery is possible. Note: 1,670 rescued rows have `RatecodeID = 99` (not a standard TLC code). Source files: `yellow_tripdata_2016-01.csv` (21.8M), `2016-02.csv` (22.8M), `2016-03.csv` (24.4M). | `RateCodeID`, `_rescued_data` | 68,999,718 | 73.02 | In Silver: `COALESCE(RateCodeID, get_json_object(_rescued_data, '$.RatecodeID'))`. Map code 99 → NULL/Unknown. Drop `_rescued_data` after recovery. |
| 2 | **Non-contiguous date range**: Data covers only Jan 2015 + Jan–Mar 2016 (4 months, 9-month gap). Seasonal analysis and year-over-year comparisons are limited. | `tpep_pickup_datetime` | — | — | Document as a known data limitation; avoid full-year seasonality claims |
| 3 | 0.60% of trips have `trip_distance = 0` (cancelled or meter-error trips) | `trip_distance` | 564,480 | 0.60 | Drop in Silver |
| 4 | 0.07% of trips have `fare_amount ≤ 0` | `fare_amount` | 62,088 | 0.07 | Drop in Silver |
| 5 | 0.04% of trips have `total_amount < 0` (voided/disputed) | `total_amount` | 34,314 | 0.04 | Drop in Silver |
| 6 | 0.02% of trips have `passenger_count = 0` | `passenger_count` | 16,428 | 0.02 | Drop or impute in Silver |
| 7 | 1,150 rows show `tip_amount > 0` on non-credit-card payments (tips should only auto-populate for credit card) | `tip_amount`, `payment_type` | 1,150 | <0.01 | Flag as suspect; keep but exclude from tip-analysis models |
| 8 | 914 trips exceed 100 miles; 458 fares exceed $500 | `trip_distance`, `fare_amount` | 1,372 | <0.01 | Cap or drop extreme outliers in Silver |
| 9 | 772 exact duplicate rows across all columns | all | 772 | <0.01 | Deduplicate in Silver |
| 10 | **GPS coordinates**: 1.55M pickup rows (1.64%) and 1.47M dropoff rows (1.55%) have (0, 0) coordinates. An additional 15K pickup / 62K dropoff rows fall outside the NYC bounding box. Non-zero ranges still span wildly (lon min −740, lat max 460). | `pickup_longitude`, `pickup_latitude`, `dropoff_longitude`, `dropoff_latitude` | \~1,545,050 | 1.64 | In Silver: NULL-out (0,0) coords and rows outside NYC bbox [−74.3, −73.7] × [40.4, 40.95]. Demand heatmaps (BQ-1) must exclude these. |
| 11 | **Trip duration anomalies**: 946 trips have negative duration (dropoff before pickup), 101,814 have zero duration, and 760,154 (0.80%) are under 1 min. On the other extreme, 126,916 trips exceed 3 hours and 330 exceed 24 hours (max \~381 days). | `tpep_pickup_datetime`, `tpep_dropoff_datetime` | 863,084 | 0.91 | In Silver: drop negative/zero durations; cap or flag trips > 3 hours. Compute `trip_duration_min` as a derived Silver column for BQ-2/BQ-3. |
| 12 | **Extreme monetary outliers**: 840 rows have negative `tip_amount` (min −$220.80), 1,368 tips exceed $100, and 2 tips exceed $1,000 (max **$3.95M**). 210 rows have `total_amount` > $1,000 (max **$3.95M**). The `extra` surcharge has 168,494 rows at $4.50 (non-standard) and various negative/irregular values. | `tip_amount`, `total_amount`, `extra` | \~2,400 | <0.01 | In Silver: cap tip at reasonable threshold (e.g. $200); drop totals > $1,000; restrict `extra` to valid set {0, 0.5, 1.0} or NULL the rest. |
| 13 | **Tip behaviour confirms cash-tip blindspot**: Credit-card trips (65.3% of all) show 96.4% tip rate with avg $2.75 / median $2.00. Cash trips (34.2%) show effectively 0% tip rate (808 of 32.3M). Tip distribution: 51.2% tip $0–2, 35.9% tip $2–5, 6.9% tip $5–10. 3.6% of credit-card trips have $0 tip. | `tip_amount`, `payment_type` | — | — | For BQ-4 and ML tip models: restrict training data to `payment_type = 1` only. The 3.6% zero-tip credit-card trips are legitimate (passenger chose $0). |

### Rescued-data recovery detail

The `_rescued_data` JSON payload contains exactly two keys:

| Key | Description |
|-----|-------------|
| `RatecodeID` | The rate code value (string). Matches valid TLC codes 1–6, plus anomalous code 99 (1,670 rows). |
| `_file_path` | Source CSV path on DBFS. Useful for lineage/debugging but not needed in Silver. |

**Recovery expression for Silver:**
```sql
COALESCE(
  RateCodeID,
  CAST(get_json_object(_rescued_data, '$.RatecodeID') AS LONG)
) AS RateCodeID
```

### Assumptions
- Bronze table is append-only (no transforms applied by I-01).
- Tip amounts for non-credit-card payments are not captured in `tip_amount` (cash tips excluded by design).
- `RateCodeID` values outside 1–6 and `payment_type` values outside 1–6 are data entry errors.
- Rows where `total_amount < 0` are voided/disputed trips and should be excluded from analytics.
- `RatecodeID = 99` in rescued data is a data entry error and should be mapped to NULL in Silver.
- GPS coordinates of (0, 0) represent missing data, not actual locations. The NYC bounding box [−74.3, −73.7] × [40.4, 40.95] is a conservative filter that includes all five boroughs and nearby airports.
- Trip durations under 1 minute are likely meter errors or cancellations. Durations over 3 hours are suspect for standard yellow cab trips (excluding JFK/Newark flat-rate rides).
- The `extra` surcharge should only be $0.50 (rush hour) or $1.00 (overnight). The $4.50 value may be a legitimate surcharge introduced in later data, but other irregular values are data errors.
- Tip prediction models (BQ-4) should only train on credit-card payments (`payment_type = 1`) since cash tips are unobservable.